# Mortgage Amortization Calculator

Computes a full month-by-month **Principal & Interest** breakdown for a fixed-rate home mortgage.

**Monthly Payment Formula:**

$$M = P \cdot \frac{r(1+r)^n}{(1+r)^n - 1}$$

| Symbol | Meaning |
|--------|---------|
| $M$ | Monthly payment (P&I) |
| $P$ | Loan principal |
| $r$ | Monthly interest rate = annual rate / 12 / 100 |
| $n$ | Total payments = years × 12 |

Each month the payment is split:
- **Interest** = remaining balance × $r$
- **Principal** = $M$ − interest
- New balance = old balance − principal paid

In [ ]:
import sys, os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Add the Tools directory so we can import mortgage_amortization
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from mortgage_amortization import MortgageCalculator

## 1. Define Mortgage Parameters & View Summary

In [ ]:
# === EDIT THESE PARAMETERS ===
LOAN_AMOUNT      = 1700000   # dollars
ANNUAL_RATE_PCT  = 6.5       # percent
TERM_YEARS       = 30        # years
EXTRA_MONTHLY    = 0         # additional principal per month
# =============================

calc = MortgageCalculator(LOAN_AMOUNT, ANNUAL_RATE_PCT, TERM_YEARS, EXTRA_MONTHLY)

summary = calc.summary()
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:>20s}: ${v:>14,.2f}")
    else:
        print(f"  {k:>20s}: {v}")

## 2. Full Amortization Schedule (DataFrame)

In [ ]:
df = calc.schedule()
print(f"Shape: {df.shape}  (rows = total payments)\n")

print("--- First 12 months ---")
display(df.head(12))

print("\n--- Last 5 months ---")
display(df.tail(5))

## 3. Principal vs Interest — Stacked Area Chart

Early in the loan, most of each payment goes to **interest**.  
Over time the split reverses and more goes to **principal**.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.stackplot(
    df["Month"],
    df["Principal"],
    df["Interest"],
    labels=["Principal", "Interest"],
    colors=["#2ecc71", "#e74c3c"],
    alpha=0.85,
)

ax.set_title("Monthly Payment Breakdown: Principal vs Interest", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Dollars ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.legend(loc="center right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Remaining Balance Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(df["Month"], df["Balance"], color="#3498db", alpha=0.3)
ax.plot(df["Month"], df["Balance"], color="#2980b9", linewidth=2)

ax.set_title("Remaining Loan Balance", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Balance ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Impact of Extra Monthly Payments

Compare the base mortgage against paying an **extra $200/month** toward principal.  
See how many months are shaved off and how much interest is saved.

In [ ]:
# Base vs extra-payment scenario
calc_extra = MortgageCalculator(LOAN_AMOUNT, ANNUAL_RATE_PCT, TERM_YEARS, extra_monthly=200)
df_extra   = calc_extra.schedule()
s_extra    = calc_extra.summary()

print("=== Comparison ===")
print(f"  Base total interest:    ${summary['total_interest']:>12,.2f}  ({summary['total_payments']} payments)")
print(f"  Extra total interest:   ${s_extra['total_interest']:>12,.2f}  ({s_extra['total_payments']} payments)")
print(f"  ────────────────────────────────────────")
print(f"  Interest saved:         ${s_extra['interest_saved']:>12,.2f}")
print(f"  Months saved:           {s_extra['months_saved']:>12d}")
print(f"  Years saved:            {s_extra['months_saved'] / 12:>12.1f}")

# Plot side-by-side balance curves
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["Month"], df["Balance"], label="Base (no extra)", linewidth=2, color="#e74c3c")
ax.plot(df_extra["Month"], df_extra["Balance"], label=f"+${200:,}/mo extra", linewidth=2, color="#2ecc71")

ax.set_title("Remaining Balance: Base vs Extra Payments", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Balance ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Annual Summary

Group the monthly schedule by **year** to see total principal and interest paid per year.

In [ ]:
# Annual aggregation
df_annual = df.copy()
df_annual["Year"] = ((df_annual["Month"] - 1) // 12) + 1

annual = (
    df_annual.groupby("Year")
    .agg(
        Payments=("Month", "count"),
        Total_Payment=("Payment", "sum"),
        Total_Principal=("Principal", "sum"),
        Total_Interest=("Interest", "sum"),
        End_Balance=("Balance", "last"),
    )
    .round(2)
)

# Format for display
annual_fmt = annual.copy()
for col in ["Total_Payment", "Total_Principal", "Total_Interest", "End_Balance"]:
    annual_fmt[col] = annual_fmt[col].apply(lambda x: f"${x:,.2f}")

display(annual_fmt)

## 7. Cumulative Interest Paid

Track how total interest accumulates over the life of the loan.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df["Month"], df["Total_Interest"], label="Base", linewidth=2, color="#e74c3c")
ax.plot(df_extra["Month"], df_extra["Total_Interest"], label="+$200/mo extra", linewidth=2, color="#2ecc71")

ax.set_title("Cumulative Interest Paid Over Time", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Cumulative Interest ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

total_base  = df["Total_Interest"].iloc[-1]
total_extra = df_extra["Total_Interest"].iloc[-1]
print(f"\nBase scenario:  ${total_base:>12,.2f} total interest")
print(f"Extra scenario: ${total_extra:>12,.2f} total interest")
print(f"Savings:        ${total_base - total_extra:>12,.2f}")

## 8. Comparative Study — Two Mortgages: $1M vs $1.7M at 5.5%

**Question:** Over 30 years at **5.5% interest**, how much do you *lose* to interest and what is the **opportunity cost** if that money were invested at **7% market return**?

| Scenario | Loan Amount | Rate | Term |
|----------|-------------|------|------|
| **A** | $1,000,000 | 5.5% | 30 yr |
| **B** | $1,700,000 | 5.5% | 30 yr |

Three perspectives per scenario:
1. **Interest Loss** — cumulative interest paid to the bank
2. **Opportunity Cost of Interest** — if each month's interest had been invested at 7%
3. **Full Opportunity Cost** — if the entire monthly P&I payment had been invested at 7%

In [ ]:
# === TWO MORTGAGE SCENARIOS ===
RATE     = 5.5    # annual interest rate for both
YEARS    = 30     # term for both
MARKET   = 7.0    # assumed annual market return
LOAN_A   = 1_000_000
LOAN_B   = 1_700_000

market_monthly = MARKET / 100.0 / 12.0

def build_study(loan_amount, rate, years):
    """Build the amortization + opportunity-cost DataFrame for one scenario."""
    calc = MortgageCalculator(loan_amount, rate, years)
    df = calc.schedule()
    ds = df[["Month", "Payment", "Principal", "Interest", "Balance"]].copy()
    ds["Cumulative_Interest_Paid"] = ds["Interest"].cumsum()

    # Interest invested at market rate
    invested = 0.0
    vals = []
    for _, row in ds.iterrows():
        invested = invested * (1 + market_monthly) + row["Interest"]
        vals.append(invested)
    ds["Interest_If_Invested"] = vals

    # Full payment invested at market rate
    invested = 0.0
    vals = []
    for _, row in ds.iterrows():
        invested = invested * (1 + market_monthly) + row["Payment"]
        vals.append(invested)
    ds["Payment_If_Invested"] = vals

    ds["Equity_Built"] = ds["Principal"].cumsum()

    for col in ["Cumulative_Interest_Paid", "Interest_If_Invested", "Payment_If_Invested", "Equity_Built"]:
        ds[col] = ds[col].round(2)
    return calc, ds

calc_a, df_a = build_study(LOAN_A, RATE, YEARS)
calc_b, df_b = build_study(LOAN_B, RATE, YEARS)

# --- Summary for both ---
def print_summary(label, loan, calc, ds):
    print(f"\n  {'─'*60}")
    print(f"  Scenario {label}:  ${loan:,.0f} at {RATE}% for {YEARS} years")
    print(f"  Monthly payment:                    ${calc.monthly_payment:>14,.2f}")
    print(f"  Total interest paid to bank:        ${ds['Cumulative_Interest_Paid'].iloc[-1]:>14,.2f}")
    print(f"  Interest $ if invested at {MARKET}%:      ${ds['Interest_If_Invested'].iloc[-1]:>14,.2f}")
    print(f"  Full payment $ if invested at {MARKET}%:  ${ds['Payment_If_Invested'].iloc[-1]:>14,.2f}")
    print(f"  Equity built (= principal repaid):  ${ds['Equity_Built'].iloc[-1]:>14,.2f}")
    opp = ds["Interest_If_Invested"].iloc[-1]
    cum = ds["Cumulative_Interest_Paid"].iloc[-1]
    print(f"  Extra cost from compounding loss:   ${opp - cum:>14,.2f}")

print("=" * 70)
print(f"  COMPARATIVE STUDY  |  Rate: {RATE}%  |  Term: {YEARS}yr  |  Market: {MARKET}%")
print("=" * 70)
print_summary("A", LOAN_A, calc_a, df_a)
print_summary("B", LOAN_B, calc_b, df_b)

# Delta
print(f"\n  {'─'*60}")
print(f"  DIFFERENCE (B − A)")
diff_interest = df_b["Cumulative_Interest_Paid"].iloc[-1] - df_a["Cumulative_Interest_Paid"].iloc[-1]
diff_opp      = df_b["Interest_If_Invested"].iloc[-1] - df_a["Interest_If_Invested"].iloc[-1]
diff_full     = df_b["Payment_If_Invested"].iloc[-1] - df_a["Payment_If_Invested"].iloc[-1]
print(f"  Extra interest paid (B vs A):        ${diff_interest:>14,.2f}")
print(f"  Extra opp cost on interest at {MARKET}%:  ${diff_opp:>14,.2f}")
print(f"  Extra opp cost on full pmt at {MARKET}%:  ${diff_full:>14,.2f}")
print("=" * 70)

In [ ]:
# --- Chart 1: Equity Built vs Interest Lost — side by side ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, ds, loan, lbl in [
    (axes[0], df_a, LOAN_A, "A"),
    (axes[1], df_b, LOAN_B, "B"),
]:
    ax.fill_between(ds["Month"], ds["Equity_Built"], color="#2ecc71", alpha=0.3)
    ax.fill_between(ds["Month"], ds["Cumulative_Interest_Paid"], color="#e74c3c", alpha=0.3)
    ax.plot(ds["Month"], ds["Equity_Built"], color="#27ae60", linewidth=2, label="Equity Built")
    ax.plot(ds["Month"], ds["Cumulative_Interest_Paid"], color="#c0392b", linewidth=2, label="Interest Paid")
    ax.set_title(f"Scenario {lbl}: ${loan:,.0f}", fontsize=13)
    ax.set_xlabel("Month")
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Dollars ($)")
fig.suptitle(f"Equity Built vs Interest Lost  —  {RATE}% / {YEARS}yr", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Opportunity Cost: Both Scenarios Overlaid

- **Solid lines**: Full payment invested at 7%
- **Dashed lines**: Interest only invested at 7%
- **Dotted lines**: Equity built (principal paid)

In [ ]:
# --- Chart 2: Opportunity Cost — Both scenarios overlaid ---
fig, ax = plt.subplots(figsize=(14, 7))

# Scenario A
ax.plot(df_a["Month"], df_a["Payment_If_Invested"],
        label=f"A (${LOAN_A/1e6:.0f}M) full pmt @ {MARKET}%", linewidth=2, color="#3498db")
ax.plot(df_a["Month"], df_a["Interest_If_Invested"],
        label=f"A (${LOAN_A/1e6:.0f}M) interest @ {MARKET}%", linewidth=2, color="#3498db", linestyle="--")
ax.plot(df_a["Month"], df_a["Equity_Built"],
        label=f"A (${LOAN_A/1e6:.0f}M) equity", linewidth=2, color="#3498db", linestyle=":")

# Scenario B
ax.plot(df_b["Month"], df_b["Payment_If_Invested"],
        label=f"B (${LOAN_B/1e6:.1f}M) full pmt @ {MARKET}%", linewidth=2, color="#e74c3c")
ax.plot(df_b["Month"], df_b["Interest_If_Invested"],
        label=f"B (${LOAN_B/1e6:.1f}M) interest @ {MARKET}%", linewidth=2, color="#e74c3c", linestyle="--")
ax.plot(df_b["Month"], df_b["Equity_Built"],
        label=f"B (${LOAN_B/1e6:.1f}M) equity", linewidth=2, color="#e74c3c", linestyle=":")

ax.set_title(f"Mortgage Costs vs {MARKET}% Market Return  —  $1M vs $1.7M at {RATE}%", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Dollars ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.legend(loc="upper left", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Year-by-Year Breakdown: Both Scenarios Side by Side

In [ ]:
# --- Annual comparative table for BOTH scenarios ---
def annual_table(ds, label, loan):
    d = ds.copy()
    d["Year"] = ((d["Month"] - 1) // 12) + 1
    agg = (
        d.groupby("Year").agg(
            Interest_Paid=("Interest", "sum"),
            Principal_Paid=("Principal", "sum"),
            Cumul_Interest=("Cumulative_Interest_Paid", "last"),
            Equity=("Equity_Built", "last"),
            Interest_Invested=("Interest_If_Invested", "last"),
            FullPmt_Invested=("Payment_If_Invested", "last"),
            Balance=("Balance", "last"),
        ).round(2)
    )
    agg["Opp_Cost_Interest"] = (agg["Interest_Invested"] - agg["Cumul_Interest"]).round(2)
    agg["Opp_Cost_Full"]     = (agg["FullPmt_Invested"] - agg["Equity"]).round(2)
    return agg

annual_a = annual_table(df_a, "A", LOAN_A)
annual_b = annual_table(df_b, "B", LOAN_B)

# Display Scenario A
print(f"Scenario A:  ${LOAN_A:,.0f} @ {RATE}%  |  Market: {MARKET}%\n")
fmt_a = annual_a.copy()
for col in fmt_a.columns:
    fmt_a[col] = fmt_a[col].apply(lambda x: f"${x:,.2f}")
display(fmt_a)

# Display Scenario B
print(f"\nScenario B:  ${LOAN_B:,.0f} @ {RATE}%  |  Market: {MARKET}%\n")
fmt_b = annual_b.copy()
for col in fmt_b.columns:
    fmt_b[col] = fmt_b[col].apply(lambda x: f"${x:,.2f}")
display(fmt_b)

### Final Verdict — Side-by-Side Bar Chart

Compare **equity built**, **interest lost**, and **market opportunity** for both the $1M and $1.7M mortgages.  
The home must appreciate enough to offset the opportunity cost.

In [ ]:
# --- Final side-by-side bar chart ---
def scenario_stats(loan, ds):
    return {
        "Equity Built":           ds["Equity_Built"].iloc[-1],
        "Interest Paid\n(Loss)":  ds["Cumulative_Interest_Paid"].iloc[-1],
        "Interest $\n@ 7% Mkt":  ds["Interest_If_Invested"].iloc[-1],
        "Full Pmt $\n@ 7% Mkt":  ds["Payment_If_Invested"].iloc[-1],
    }

stats_a = scenario_stats(LOAN_A, df_a)
stats_b = scenario_stats(LOAN_B, df_b)

categories = list(stats_a.keys())
vals_a = list(stats_a.values())
vals_b = list(stats_b.values())

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 7))
bars_a = ax.bar(x - width/2, vals_a, width, label=f"A: ${LOAN_A/1e6:.0f}M", color="#3498db", edgecolor="white")
bars_b = ax.bar(x + width/2, vals_b, width, label=f"B: ${LOAN_B/1e6:.1f}M", color="#e74c3c", edgecolor="white")

for bars in [bars_a, bars_b]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 15000,
                f"${h:,.0f}", ha="center", va="bottom", fontweight="bold", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylabel("Dollars ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.set_title(f"30-Year Mortgage at {RATE}%  vs  {MARKET}% Market Returns", fontsize=14)
ax.legend(fontsize=12)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Breakeven appreciation rates
be_a = ((df_a["Payment_If_Invested"].iloc[-1] / LOAN_A) ** (1.0/YEARS) - 1) * 100
be_b = ((df_b["Payment_If_Invested"].iloc[-1] / LOAN_B) ** (1.0/YEARS) - 1) * 100

print(f"\n{'='*70}")
print(f"  TO BREAK EVEN WITH {MARKET}% MARKET RETURNS:")
print(f"  Scenario A (${LOAN_A/1e6:.0f}M):  home must appreciate ≥ {be_a:.2f}%/yr")
print(f"  Scenario B (${LOAN_B/1e6:.1f}M):  home must appreciate ≥ {be_b:.2f}%/yr")
print(f"{'='*70}")

## 9. $700K Allocation Study — Mortgage Paydown vs Market Investment

You have **$700,000** available and a **$1.7M mortgage at 5.5% / 30yr**.  
Should you use it to pay down the mortgage, invest in the market at 7%, or split it?

| Scenario | To Mortgage | To Market | Remaining Loan |
|----------|-------------|-----------|---------------|
| **1. All Market** | $0 | $700K | $1,700,000 |
| **2. All Mortgage** | $700K | $0 | $1,000,000 |
| **3. Optimal Split** | $X | $700K − X | $1,700,000 − X |

**Net Worth at year 30** = Market Portfolio Value + Home Equity − Remaining Balance

We sweep X from $0 to $700K in $10K steps to find the allocation that **maximizes 30-year net worth**.

In [ ]:
# === ALLOCATION PARAMETERS ===
LUMP_SUM    = 700_000       # cash available
MORTGAGE    = 1_700_000     # original loan
MORT_RATE   = 6           # mortgage annual rate %
MORT_YEARS  = 30            # term
MKT_RATE    = 7.0           # expected annual market return %

mort_r   = MORT_RATE / 100.0 / 12.0
mkt_r    = MKT_RATE  / 100.0 / 12.0
n_months = MORT_YEARS * 12

def scenario_net_worth(to_mortgage: float, to_market: float, reinvest_savings: bool = True):
    """
    Compute 30-year net worth given a lump-sum split.
    
    - to_mortgage: applied as Day-1 principal reduction
    - to_market:   invested Day-1 and compounds monthly at MKT_RATE
    - reinvest_savings: if True, the monthly payment savings (vs full mortgage)
      are reinvested into the market each month (DCA strategy).
      If False, savings are NOT reinvested — they simply disappear (spent/kept as cash).
    
    Returns dict with detailed breakdown.
    """
    loan = MORTGAGE - to_mortgage
    
    # Monthly P&I on the reduced loan
    if mort_r == 0:
        monthly_pmt = loan / n_months
    else:
        factor = (1 + mort_r) ** n_months
        monthly_pmt = loan * (mort_r * factor) / (factor - 1)
    
    # --- Amortize the mortgage month by month ---
    balance = loan
    total_interest = 0.0
    total_principal = 0.0
    
    # Full mortgage payment (no lump sum) — baseline cash outflow
    factor_full = (1 + mort_r) ** n_months
    full_monthly = MORTGAGE * (mort_r * factor_full) / (factor_full - 1)
    
    market_value = to_market
    monthly_saving = full_monthly - monthly_pmt  # extra cash flow vs keeping full mortgage
    
    # Monthly DCA contribution: only if reinvest_savings is True
    monthly_dca = monthly_saving if reinvest_savings else 0.0
    
    for m in range(1, n_months + 1):
        # Mortgage
        interest = balance * mort_r
        principal = monthly_pmt - interest
        if principal > balance:
            principal = balance
        balance -= principal
        total_interest += interest
        total_principal += principal
        
        # Market: compound existing + add monthly DCA (if enabled)
        market_value = market_value * (1 + mkt_r) + monthly_dca
        
        if balance <= 0:
            # Loan paid off — remaining payments all go to market
            remaining = n_months - m
            dca_after_payoff = full_monthly if reinvest_savings else 0.0
            for _ in range(remaining):
                market_value = market_value * (1 + mkt_r) + dca_after_payoff
            balance = 0
            break
    
    equity = MORTGAGE  # you own the full home value (equal to original price for simplicity)
    net_worth = market_value + equity - max(balance, 0)
    
    return {
        "to_mortgage": to_mortgage,
        "to_market": to_market,
        "reduced_loan": MORTGAGE - to_mortgage,
        "monthly_payment": round(monthly_pmt, 2),
        "monthly_saving": round(monthly_saving, 2),
        "monthly_dca": round(monthly_dca, 2),
        "reinvest_savings": reinvest_savings,
        "total_interest": round(total_interest, 2),
        "market_value_30yr": round(market_value, 2),
        "remaining_balance": round(max(balance, 0), 2),
        "equity": equity,
        "net_worth_30yr": round(net_worth, 2),
    }

# =============================================
# SCENARIO 1: All $700K in market, full mortgage
#             (no savings to reinvest — payment stays the same)
# =============================================
s1 = scenario_net_worth(0, LUMP_SUM, reinvest_savings=True)

# =============================================
# SCENARIO 2: All $700K to mortgage, savings NOT reinvested
#             (lower payment, but extra cash just sits/spent)
# =============================================
s2 = scenario_net_worth(LUMP_SUM, 0, reinvest_savings=False)

# =============================================
# SCENARIO 3: All $700K to mortgage, savings DCA'd into market
#             (lower payment, AND the difference goes to market monthly)
# =============================================
s3 = scenario_net_worth(LUMP_SUM, 0, reinvest_savings=True)

# =============================================
# SCENARIO 5: $500K → Market, $200K → Mortgage, with DCA
#             (hybrid split with DCA on payment savings)
# =============================================
S5_TO_MARKET   = 500_000
S5_TO_MORTGAGE = 200_000
s5 = scenario_net_worth(S5_TO_MORTGAGE, S5_TO_MARKET, reinvest_savings=True)

# =============================================
# SCENARIO 4: Sweep allocations to find optimal (with DCA)
# =============================================
steps = range(0, LUMP_SUM + 1, 10_000)  # $10K increments
results = [scenario_net_worth(x, LUMP_SUM - x, reinvest_savings=True) for x in steps]
df_alloc = pd.DataFrame(results)

# Find optimal
best = df_alloc.loc[df_alloc["net_worth_30yr"].idxmax()]

# =============================================
# PRINT COMPARISON
# =============================================
def fmt(label, s):
    print(f"\n  {'─'*65}")
    print(f"  {label}")
    print(f"  {'─'*65}")
    print(f"    To mortgage (Day 1):      ${s['to_mortgage']:>14,.2f}")
    print(f"    To market (Day 1):        ${s['to_market']:>14,.2f}")
    print(f"    Reduced loan:             ${s['reduced_loan']:>14,.2f}")
    print(f"    Monthly payment:          ${s['monthly_payment']:>14,.2f}")
    print(f"    Monthly saving vs full:   ${s['monthly_saving']:>14,.2f}")
    print(f"    Monthly DCA to market:    ${s['monthly_dca']:>14,.2f}")
    print(f"    Reinvest savings?         {'YES' if s['reinvest_savings'] else 'NO':>14s}")
    print(f"    Total interest (30yr):    ${s['total_interest']:>14,.2f}")
    print(f"    Market portfolio (30yr):  ${s['market_value_30yr']:>14,.2f}")
    print(f"    Home equity:              ${s['equity']:>14,.2f}")
    print(f"    ══════════════════════════════════════")
    print(f"    NET WORTH at Year 30:     ${s['net_worth_30yr']:>14,.2f}")

print("=" * 72)
print(f"  $700K ALLOCATION STUDY  |  Mortgage: ${MORTGAGE:,.0f} @ {MORT_RATE}%")
print(f"  Market return: {MKT_RATE}%  |  Term: {MORT_YEARS} years")
print("=" * 72)

fmt("S1: All $700K → Market, Keep Full $1.7M Mortgage", s1)
fmt("S2: All $700K → Mortgage, Savings NOT Reinvested", s2)
fmt("S3: All $700K → Mortgage, Savings DCA'd into Market", s3)
fmt(f"S5: ${S5_TO_MARKET:,.0f} → Market, ${S5_TO_MORTGAGE:,.0f} → Mortgage (with DCA)", s5)
fmt(f"Opt (OPTIMAL): ${best['to_mortgage']:,.0f} → Mortgage, ${best['to_market']:,.0f} → Market (with DCA)", best)

# Winner
print(f"\n{'='*72}")
nw_all = {"S1": s1['net_worth_30yr'], "S2": s2['net_worth_30yr'],
           "S3": s3['net_worth_30yr'], "S5": s5['net_worth_30yr'],
           "Opt": best['net_worth_30yr']}
winner = max(nw_all, key=nw_all.get)
print(f"  WINNER: {winner} with ${nw_all[winner]:,.0f}")
print(f"  ────────────────────────────────────────")
print(f"  S1 (all market)          : ${s1['net_worth_30yr']:>14,.0f}")
print(f"  S2 (all mortgage, no DCA): ${s2['net_worth_30yr']:>14,.0f}")
print(f"  S3 (all mortgage + DCA)  : ${s3['net_worth_30yr']:>14,.0f}")
print(f"  S5 ($500K mkt + $200K mtg): ${s5['net_worth_30yr']:>14,.0f}")
print(f"  Opt (optimal split + DCA): ${best['net_worth_30yr']:>14,.0f}")
print(f"  ────────────────────────────────────────")
print(f"  S1 vs S2:  ${s1['net_worth_30yr'] - s2['net_worth_30yr']:>+14,.0f}  ({'Market' if s1['net_worth_30yr'] > s2['net_worth_30yr'] else 'Mortgage'} wins)")
print(f"  S1 vs S3:  ${s1['net_worth_30yr'] - s3['net_worth_30yr']:>+14,.0f}  ({'Market' if s1['net_worth_30yr'] > s3['net_worth_30yr'] else 'Mortgage+DCA'} wins)")
print(f"  S1 vs S5:  ${s1['net_worth_30yr'] - s5['net_worth_30yr']:>+14,.0f}  ({'S1' if s1['net_worth_30yr'] > s5['net_worth_30yr'] else 'S5'} wins)")
print(f"  S5 vs S3:  ${s5['net_worth_30yr'] - s3['net_worth_30yr']:>+14,.0f}  ({'S5' if s5['net_worth_30yr'] > s3['net_worth_30yr'] else 'S3'} wins)")
print(f"  S3 vs S2:  ${s3['net_worth_30yr'] - s2['net_worth_30yr']:>+14,.0f}  (DCA value)")
int_saved = s1['total_interest'] - s3['total_interest']
print(f"  Interest saved (S3 vs S1): ${int_saved:>14,.0f}")
print(f"{'='*72}")

In [ ]:
# --- Chart: Net Worth vs Allocation Split ---
fig, ax1 = plt.subplots(figsize=(14, 7))

# Net worth curve (with DCA reinvestment)
ax1.plot(df_alloc["to_mortgage"] / 1000, df_alloc["net_worth_30yr"] / 1e6,
         linewidth=3, color="#2ecc71", label="Net Worth at Year 30 (with DCA)")
ax1.axvline(best["to_mortgage"] / 1000, color="#e74c3c", linestyle="--", linewidth=1.5,
            label=f"Optimal: ${best['to_mortgage']:,.0f} to mortgage")
ax1.scatter([best["to_mortgage"] / 1000], [best["net_worth_30yr"] / 1e6],
            color="#e74c3c", s=120, zorder=5)

# Mark S1: All Market
ax1.scatter([0], [s1["net_worth_30yr"] / 1e6], color="#3498db", s=120, zorder=5, marker="D")
ax1.annotate(f"S1: All Market\n${s1['net_worth_30yr']/1e6:.2f}M",
             xy=(0, s1["net_worth_30yr"]/1e6), xytext=(60, s1["net_worth_30yr"]/1e6 - 0.25),
             fontsize=10, fontweight="bold", color="#3498db",
             arrowprops=dict(arrowstyle="->", color="#3498db"))

# Mark S2: All Mortgage, no DCA
ax1.scatter([LUMP_SUM / 1000], [s2["net_worth_30yr"] / 1e6], color="#e67e22", s=120, zorder=5, marker="s")
ax1.annotate(f"S2: Mortgage (no DCA)\n${s2['net_worth_30yr']/1e6:.2f}M",
             xy=(LUMP_SUM/1000, s2["net_worth_30yr"]/1e6),
             xytext=(LUMP_SUM/1000 - 200, s2["net_worth_30yr"]/1e6 - 0.15),
             fontsize=10, fontweight="bold", color="#e67e22",
             arrowprops=dict(arrowstyle="->", color="#e67e22"))

# Mark S3: All Mortgage + DCA
ax1.scatter([LUMP_SUM / 1000], [s3["net_worth_30yr"] / 1e6], color="#9b59b6", s=120, zorder=5, marker="D")
ax1.annotate(f"S3: Mortgage + DCA\n${s3['net_worth_30yr']/1e6:.2f}M",
             xy=(LUMP_SUM/1000, s3["net_worth_30yr"]/1e6),
             xytext=(LUMP_SUM/1000 - 200, s3["net_worth_30yr"]/1e6 + 0.15),
             fontsize=10, fontweight="bold", color="#9b59b6",
             arrowprops=dict(arrowstyle="->", color="#9b59b6"))

# Mark S5: $500K Market + $200K Mortgage + DCA
ax1.scatter([S5_TO_MORTGAGE / 1000], [s5["net_worth_30yr"] / 1e6], color="#1abc9c", s=120, zorder=5, marker="^")
ax1.annotate(f"S5: $500K Mkt + $200K Mtg\n${s5['net_worth_30yr']/1e6:.2f}M",
             xy=(S5_TO_MORTGAGE/1000, s5["net_worth_30yr"]/1e6),
             xytext=(S5_TO_MORTGAGE/1000 + 80, s5["net_worth_30yr"]/1e6 - 0.20),
             fontsize=10, fontweight="bold", color="#1abc9c",
             arrowprops=dict(arrowstyle="->", color="#1abc9c"))

ax1.set_xlabel("Amount Applied to Mortgage ($K)", fontsize=12)
ax1.set_ylabel("Net Worth at Year 30 ($M)", fontsize=12)
ax1.set_title(f"$700K Allocation: Mortgage Paydown vs Market Investment\n"
              f"Mortgage {MORT_RATE}% | Market {MKT_RATE}% | 30 Years", fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.2f}M"))

plt.tight_layout()
plt.show()

# --- Second chart: Components breakdown ---
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_alloc["to_mortgage"] / 1000, df_alloc["market_value_30yr"] / 1e6,
        linewidth=2, color="#3498db", label="Market Portfolio Value")
ax.plot(df_alloc["to_mortgage"] / 1000, df_alloc["total_interest"] / 1e6,
        linewidth=2, color="#e74c3c", label="Total Interest Paid")

# Use twin axis for monthly payment
ax2 = ax.twinx()
ax2.plot(df_alloc["to_mortgage"] / 1000, df_alloc["monthly_payment"],
         linewidth=2, color="#9b59b6", linestyle="--", label="Monthly Payment")
ax2.set_ylabel("Monthly Payment ($)", color="#9b59b6", fontsize=12)
ax2.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax2.tick_params(axis="y", labelcolor="#9b59b6")

ax.set_xlabel("Amount Applied to Mortgage ($K)", fontsize=12)
ax.set_ylabel("Dollars ($M)", fontsize=12)
ax.set_title("Components: Market Portfolio & Interest Paid vs Allocation", fontsize=14)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.2f}M"))
ax.legend(loc="upper right", fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### --- Summary Table: Year-by-year milestones (1–30) ---


In [ ]:
# --- Summary Table: Year-by-year milestones (1–30) ---
milestones = list(range(1, 31))
rows = []

def snapshot(to_mtg, to_mkt, months, reinvest):
    """Compute net worth at a given month for one allocation."""
    loan = MORTGAGE - to_mtg
    factor = (1 + mort_r) ** n_months
    monthly_pmt = loan * (mort_r * factor) / (factor - 1)
    full_factor = (1 + mort_r) ** n_months
    full_monthly = MORTGAGE * (mort_r * full_factor) / (full_factor - 1)
    saving = full_monthly - monthly_pmt
    monthly_dca = saving if reinvest else 0.0
    
    bal = loan
    tot_int = 0
    mkt = to_mkt
    for i in range(1, months + 1):
        interest = bal * mort_r
        princ = monthly_pmt - interest
        if princ > bal:
            princ = bal
        bal -= princ
        tot_int += interest
        mkt = mkt * (1 + mkt_r) + monthly_dca
        if bal <= 0:
            remaining = months - i
            dca_after = full_monthly if reinvest else 0.0
            for _ in range(remaining):
                mkt = mkt * (1 + mkt_r) + dca_after
            bal = 0
            break
    
    nw = mkt + MORTGAGE - max(bal, 0)
    return round(nw, 2), round(mkt, 2), round(tot_int, 2), round(max(bal, 0), 2)

for yr in milestones:
    m = yr * 12
    nw1, mkt1, int1, bal1 = snapshot(0, LUMP_SUM, m, reinvest=True)
    nw2, mkt2, int2, bal2 = snapshot(LUMP_SUM, 0, m, reinvest=False)
    nw3, mkt3, int3, bal3 = snapshot(LUMP_SUM, 0, m, reinvest=True)
    nw4, mkt4, int4, bal4 = snapshot(best["to_mortgage"], best["to_market"], m, reinvest=True)
    nw5, mkt5, int5, bal5 = snapshot(S5_TO_MORTGAGE, S5_TO_MARKET, m, reinvest=True)
    
    rows.append({
        "Year": yr,
        "S1_NW": nw1, "S1_Market": mkt1, "S1_Interest": int1,
        "S2_NW": nw2, "S2_Market": mkt2, "S2_Interest": int2,
        "S3_NW": nw3, "S3_Market": mkt3, "S3_Interest": int3,
        "S5_NW": nw5, "S5_Market": mkt5, "S5_Interest": int5,
        "Opt_NW": nw4, "Opt_Market": mkt4,
        "S1_vs_S2": round(nw1 - nw2, 2),
        "S1_vs_S3": round(nw1 - nw3, 2),
        "S1_vs_S5": round(nw1 - nw5, 2),
        "S3_vs_S2": round(nw3 - nw2, 2),
    })

df_mile = pd.DataFrame(rows).set_index("Year")

# Format
print(f"Net Worth Comparison — Year by Year (1–30)")
print(f"S1 = All Market | S2 = Mortgage (no DCA) | S3 = Mortgage + DCA | S5 = $500K Mkt + $200K Mtg | Opt = ${best['to_mortgage']:,.0f} split\n")

fmt_mile = df_mile.copy()
for col in fmt_mile.columns:
    fmt_mile[col] = fmt_mile[col].apply(lambda x: f"${x:,.0f}")
display(fmt_mile)

# --- Column Legend ---
print(f"\n{'━'*90}")
print("  COLUMN LEGEND")
print(f"{'━'*90}")
print("  S1_NW         Net worth: all $700K in market, full $1.7M mortgage")
print("  S1_Market      Market portfolio value under S1")
print("  S1_Interest    Cumulative mortgage interest paid under S1")
print()
print("  S2_NW         Net worth: all $700K to mortgage, savings NOT reinvested")
print("  S2_Market      Market portfolio (stays $0 — no lump sum, no DCA)")
print("  S2_Interest    Cumulative mortgage interest paid under S2")
print()
print("  S3_NW         Net worth: all $700K to mortgage, monthly savings DCA'd into market")
print("  S3_Market      Market portfolio (funded by monthly payment savings reinvested)")
print("  S3_Interest    Cumulative mortgage interest paid under S3 (same as S2)")
print()
print("  S5_NW         Net worth: $500K market + $200K mortgage prepay + DCA")
print("  S5_Market      Market portfolio under S5 ($500K lump + DCA from $200K savings)")
print("  S5_Interest    Cumulative mortgage interest paid under S5")
print()
print("  Opt_NW        Net worth under the optimal split (with DCA)")
print("  Opt_Market     Market portfolio under the optimal split")
print()
print("  S1_vs_S2       S1 − S2  (positive = all-market wins vs mortgage without DCA)")
print("  S1_vs_S3       S1 − S3  (positive = all-market wins vs mortgage with DCA)")
print("  S1_vs_S5       S1 − S5  (positive = all-market wins vs hybrid split)")
print("  S3_vs_S2       S3 − S2  (value added by reinvesting monthly savings)")
print(f"{'━'*90}")
print(f"  Net Worth = Market Portfolio + Home Equity ($1.7M) − Remaining Balance")
print(f"  DCA = Dollar-Cost Averaging (monthly savings → market investment)")
print(f"{'━'*90}")

### WINNER ANALYSIS BY TIME HORIZON + OPPORTUNITY LOSS vs BEST STRATEGY


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WINNER ANALYSIS BY TIME HORIZON + OPPORTUNITY LOSS vs BEST STRATEGY
# ══════════════════════════════════════════════════════════════════════════════

# --- 1. Determine winner at each year ---
strategy_names = {
    "S1": "All Market",
    "S2": "Mortgage (no DCA)",
    "S3": "Mortgage + DCA",
    "S5": "$500K Mkt + $200K Mtg",
    "Opt": f"Optimal (${best['to_mortgage']:,.0f} split)",
}

winner_rows = []
for yr in range(1, 31):
    m = yr * 12
    nw1, _, _, _ = snapshot(0, LUMP_SUM, m, reinvest=True)
    nw2, _, _, _ = snapshot(LUMP_SUM, 0, m, reinvest=False)
    nw3, _, _, _ = snapshot(LUMP_SUM, 0, m, reinvest=True)
    nw5, _, _, _ = snapshot(S5_TO_MORTGAGE, S5_TO_MARKET, m, reinvest=True)
    nw4, _, _, _ = snapshot(best["to_mortgage"], best["to_market"], m, reinvest=True)
    
    nw_map = {"S1": nw1, "S2": nw2, "S3": nw3, "S5": nw5, "Opt": nw4}
    winner_key = max(nw_map, key=nw_map.get)
    best_nw = nw_map[winner_key]
    
    row = {
        "Year": yr,
        "S1_NW": nw1,
        "S2_NW": nw2,
        "S3_NW": nw3,
        "S5_NW": nw5,
        "Opt_NW": nw4,
        "Winner": f"{winner_key} ({strategy_names[winner_key]})",
        "Best_NW": best_nw,
        "S1 (All Market) Loss": round(nw1 - best_nw, 2),
        "S2 (All Mortgage) Loss": round(nw2 - best_nw, 2),
        "S3 (Mortgage+DCA) Loss": round(nw3 - best_nw, 2),
        "S5 ($500K+$200K) Loss": round(nw5 - best_nw, 2),
        "Opt (Best Split) Loss": round(nw4 - best_nw, 2),
    }
    winner_rows.append(row)

df_winners = pd.DataFrame(winner_rows).set_index("Year")

# --- 2. Highlight winners at key horizons ---
key_horizons = [5, 10, 15, 20, 25, 30]

print("=" * 100)
print("  WINNER BY TIME HORIZON")
print("=" * 100)
for yr in key_horizons:
    row = df_winners.loc[yr]
    print(f"\n  Year {yr:2d}:  {row['Winner']}")
    print(f"          S1: ${row['S1_NW']:>14,.0f}   S2: ${row['S2_NW']:>14,.0f}   "
          f"S3: ${row['S3_NW']:>14,.0f}   S5: ${row['S5_NW']:>14,.0f}   Opt: ${row['Opt_NW']:>14,.0f}")

# Check if winner changes over time
print(f"\n{'─'*100}")
prev_winner = None
for yr in range(1, 31):
    w = df_winners.loc[yr, "Winner"]
    if w != prev_winner:
        print(f"  Year {yr:2d}: Winner changes to → {w}")
        prev_winner = w

# --- 3. Opportunity Loss DataFrame ---
# For each strategy, how much you LOSE vs the best strategy at that year
print(f"\n{'═'*100}")
print("  OPPORTUNITY LOSS vs BEST STRATEGY (negative = you lose this much)")
print(f"{'═'*100}\n")

loss_cols = ["S1 (All Market) Loss", "S2 (All Mortgage) Loss",
             "S3 (Mortgage+DCA) Loss", "S5 ($500K+$200K) Loss",
             "Opt (Best Split) Loss"]
df_opp_loss = df_winners[["Winner"] + loss_cols].copy()

fmt_opp = df_opp_loss.copy()
for col in loss_cols:
    fmt_opp[col] = fmt_opp[col].apply(lambda x: f"${x:+,.0f}" if x != 0 else "← BEST")
display(fmt_opp)

# --- 4. Summary: Different winners at 5, 10, 15 yr ---
print(f"\n{'━'*100}")
print("  STRATEGY RECOMMENDATION BY HORIZON")
print(f"{'━'*100}")

for yr in [5, 10, 15]:
    row = df_winners.loc[yr]
    nw_map = {"S1": row["S1_NW"], "S2": row["S2_NW"], "S3": row["S3_NW"],
              "S5": row["S5_NW"], "Opt": row["Opt_NW"]}
    ranked = sorted(nw_map.items(), key=lambda x: -x[1])
    winner_key = ranked[0][0]
    runner_up = ranked[1]
    gap = ranked[0][1] - runner_up[1]
    
    print(f"\n  ┌─ {yr}-YEAR WINNER: {winner_key} ({strategy_names[winner_key]})")
    print(f"  │  Net Worth: ${ranked[0][1]:,.0f}")
    print(f"  │  Runner-up: {runner_up[0]} ({strategy_names[runner_up[0]]}) — ${runner_up[1]:,.0f}  (gap: ${gap:,.0f})")
    print(f"  │  Ranking:")
    for i, (k, v) in enumerate(ranked, 1):
        loss = v - ranked[0][1]
        print(f"  │    {i}. {k:4s} ({strategy_names[k]:24s})  ${v:>14,.0f}  {f'({loss:+,.0f})' if loss < 0 else ''}")
    print(f"  └{'─'*80}")

# Overall insight
print(f"\n{'━'*100}")
w5  = df_winners.loc[5, "Winner"]
w10 = df_winners.loc[10, "Winner"]
w15 = df_winners.loc[15, "Winner"]
w30 = df_winners.loc[30, "Winner"]

if w5 == w10 == w15 == w30:
    print(f"  CONCLUSION: {w5} dominates at ALL time horizons.")
else:
    unique = []
    for yr, w in [(5, w5), (10, w10), (15, w15), (30, w30)]:
        unique.append(f"Yr{yr}: {w}")
    print(f"  CONCLUSION: Winner varies by horizon — {' | '.join(unique)}")
    print(f"  Your optimal strategy depends on your investment time horizon!")
print(f"{'━'*100}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# KEY MILESTONES: 5, 10, and 15 YEARS — Detailed Breakdown
# ══════════════════════════════════════════════════════════════════════════════

milestone_years = [5, 10, 15]
milestone_detail_rows = []

for yr in milestone_years:
    m = yr * 12
    nw1, mkt1, int1, bal1 = snapshot(0, LUMP_SUM, m, reinvest=True)
    nw2, mkt2, int2, bal2 = snapshot(LUMP_SUM, 0, m, reinvest=False)
    nw3, mkt3, int3, bal3 = snapshot(LUMP_SUM, 0, m, reinvest=True)
    nw5, mkt5, int5, bal5 = snapshot(S5_TO_MORTGAGE, S5_TO_MARKET, m, reinvest=True)
    nw4, mkt4, int4, bal4 = snapshot(best["to_mortgage"], best["to_market"], m, reinvest=True)

    for key, label, nw, mkt, interest, bal in [
        ("S1", "All Market",              nw1, mkt1, int1, bal1),
        ("S2", "Mortgage (no DCA)",       nw2, mkt2, int2, bal2),
        ("S3", "Mortgage + DCA",          nw3, mkt3, int3, bal3),
        ("S5", "$500K Mkt + $200K Mtg",   nw5, mkt5, int5, bal5),
        ("Opt", "Optimal Split",          nw4, mkt4, int4, bal4),
    ]:
        milestone_detail_rows.append({
            "Year": yr,
            "Strategy": f"{key} ({label})",
            "Net Worth": nw,
            "Market Portfolio": mkt,
            "Interest Paid": interest,
            "Remaining Balance": bal,
            "Home Equity": MORTGAGE,
        })

df_ms = pd.DataFrame(milestone_detail_rows)

# ── Display per milestone ──
for yr in milestone_years:
    sub = df_ms[df_ms["Year"] == yr].copy()
    sub = sub.set_index("Strategy")
    sub = sub.drop(columns=["Year"])

    # Find winner
    winner_idx = sub["Net Worth"].idxmax()
    best_nw_val = sub.loc[winner_idx, "Net Worth"]

    # Add opportunity loss column
    sub["Opp Loss vs Best"] = sub["Net Worth"] - best_nw_val

    print(f"\n{'═'*90}")
    print(f"  {yr}-YEAR MILESTONE  |  Winner: {winner_idx}")
    print(f"{'═'*90}\n")

    fmt_sub = sub.copy()
    for col in fmt_sub.columns:
        if col == "Opp Loss vs Best":
            fmt_sub[col] = fmt_sub[col].apply(
                lambda x: "← BEST" if x == 0 else f"${x:+,.0f}"
            )
        else:
            fmt_sub[col] = fmt_sub[col].apply(lambda x: f"${x:,.0f}")
    display(fmt_sub)

# ── Summary: Winners across milestones ──
print(f"\n{'━'*90}")
print("  MILESTONE WINNERS SUMMARY")
print(f"{'━'*90}")

winner_summary_rows = []
for yr in milestone_years:
    sub = df_ms[df_ms["Year"] == yr].set_index("Strategy")
    winner_strat = sub["Net Worth"].idxmax()
    best_val = sub.loc[winner_strat, "Net Worth"]
    ranked = sub["Net Worth"].sort_values(ascending=False)
    runner_up = ranked.index[1]
    gap = best_val - ranked.iloc[1]

    winner_summary_rows.append({
        "Horizon": f"{yr} Years",
        "Winner": winner_strat,
        "Net Worth": best_val,
        "Runner-Up": runner_up,
        "Runner-Up NW": ranked.iloc[1],
        "Lead ($)": gap,
    })

    print(f"\n  Year {yr:2d}: {winner_strat}")
    print(f"          Net Worth: ${best_val:,.0f}  |  Lead over runner-up: ${gap:,.0f}")

df_winner_summary = pd.DataFrame(winner_summary_rows).set_index("Horizon")

# Format and display
fmt_ws = df_winner_summary.copy()
for col in ["Net Worth", "Runner-Up NW", "Lead ($)"]:
    fmt_ws[col] = fmt_ws[col].apply(lambda x: f"${x:,.0f}")

print(f"\n")
display(fmt_ws)

# Check if winner is consistent
winners_set = set(df_winner_summary["Winner"])
if len(winners_set) == 1:
    print(f"\n  ✓ Same winner ({list(winners_set)[0]}) at all milestones — consistent strategy.")
else:
    print(f"\n  ⚠ Winner changes across horizons — your decision depends on your timeline!")
    for _, row in df_winner_summary.iterrows():
        print(f"    {row.name}: {row['Winner']}")
print(f"{'━'*90}")

*(Excel export moved to the final cell)*

In [ ]:
# Excel export moved to the final cell of the notebook

## 11. Split + DCA Sensitivity: Which Hybrid Beats S1?

Sweep **every possible Market / Mortgage split** (in $10K steps) **with DCA enabled**, and compare each variant against S1 (All Market) at multiple time horizons (5, 10, 15, 20, 25, 30 years).

Key questions answered:
- At which split does the hybrid strategy **first beat S1**?
- Which split is **optimal at each horizon**?
- How does the **advantage/disadvantage** change over time?

## Section 12 — Peace-of-Mind Constraint: Monthly Payment ≤ $5,000

**Question:** If I want to keep my monthly mortgage payment at or under **$5,000** for peace of mind,
what is the minimum lump-sum I must put toward the mortgage, and given that constraint,
what is the best split of the remaining $700K?

- Full $1.7M mortgage at 6% / 30yr → monthly payment ≈ $10,189
- Target: ≤ $5,000/month → requires a meaningful down payment from the $700K

In [ ]:
"""
Section 12 — Peace-of-Mind: Monthly Payment ≤ $5,000
=====================================================
Constraint: monthly mortgage outgo must not exceed $5,000.
Given $1.7M mortgage at 6% / 30yr, find the minimum down payment from
the $700K lump sum, then sweep remaining splits to find the best strategy.
"""

# Helper: monthly payment calculator
def _monthly_pmt(principal, r=mort_r, n=n_months):
    if principal <= 0:
        return 0.0
    factor = (1 + r) ** n
    return principal * (r * factor) / (factor - 1)

# S1 baseline at each horizon
horizons = [5, 10, 15, 20, 25, 30]
_full = _monthly_pmt(MORTGAGE)
s1_by_horizon = {}
for _yr in horizons:
    _m = _yr * 12
    _mkt = LUMP_SUM * (1 + mkt_r) ** _m
    _bal = MORTGAGE * (1 + mort_r) ** _m - _full * ((1 + mort_r) ** _m - 1) / mort_r
    s1_by_horizon[_yr] = _mkt + MORTGAGE - max(_bal, 0)

MAX_MONTHLY = 5_000  # peace-of-mind cap

# ── 1. Find minimum down payment to stay ≤ $5K/mo ──
full_pmt = _monthly_pmt(MORTGAGE)
print(f"Full $1.7M mortgage payment: ${full_pmt:,.2f}/mo")
print(f"Target max payment:          ${MAX_MONTHLY:,.2f}/mo\n")

# Sweep $1K increments to find minimum down payment
min_to_mortgage = 0
for amt in range(0, LUMP_SUM + 1, 1_000):
    pmt_check = _monthly_pmt(MORTGAGE - amt)
    if pmt_check <= MAX_MONTHLY:
        min_to_mortgage = amt
        break

remaining_mortgage = MORTGAGE - min_to_mortgage
actual_pmt = _monthly_pmt(remaining_mortgage)
remaining_lump = LUMP_SUM - min_to_mortgage

print("═" * 90)
print(f"  MINIMUM DOWN PAYMENT TO ACHIEVE ≤ ${MAX_MONTHLY:,}/mo")
print("═" * 90)
print(f"  Minimum to mortgage:   ${min_to_mortgage:>12,}")
print(f"  Remaining mortgage:    ${remaining_mortgage:>12,}")
print(f"  Actual monthly payment: ${actual_pmt:>11,.2f}")
print(f"  Remaining to invest:   ${remaining_lump:>12,}")
print(f"  Monthly savings vs full mortgage: ${full_pmt - actual_pmt:,.2f}/mo")
print("═" * 90)

# ── 2. Sweep: given the constraint, how to split the remaining cash ──
constrained_rows = []
for extra_to_mtg in range(0, remaining_lump + 1, 10_000):
    total_to_mtg = min_to_mortgage + extra_to_mtg
    to_mkt = LUMP_SUM - total_to_mtg
    reduced_mortgage = MORTGAGE - total_to_mtg
    monthly_pmt = _monthly_pmt(reduced_mortgage)
    dca = full_pmt - monthly_pmt  # savings vs full mortgage payment

    for yr in horizons:
        m = yr * 12
        # Market portfolio: lump sum + DCA annuity
        mkt_val = to_mkt * (1 + mkt_r) ** m
        if dca > 0:
            mkt_val += dca * ((1 + mkt_r) ** m - 1) / mkt_r

        # Mortgage balance at month m
        if reduced_mortgage > 0:
            bal = reduced_mortgage * (1 + mort_r) ** m - monthly_pmt * ((1 + mort_r) ** m - 1) / mort_r
            bal = max(bal, 0)
        else:
            bal = 0

        nw = mkt_val + MORTGAGE - bal
        constrained_rows.append({
            "Extra to Mortgage ($K)": extra_to_mtg / 1000,
            "Total to Mortgage ($K)": total_to_mtg / 1000,
            "To Market ($K)": to_mkt / 1000,
            "Monthly Payment ($)": round(monthly_pmt, 2),
            "DCA Savings ($/mo)": round(dca, 2),
            "Horizon (yr)": yr,
            "Net Worth ($)": round(nw, 2),
            "vs S1 ($)": round(nw - s1_by_horizon[yr], 2),
        })

df_constrained = pd.DataFrame(constrained_rows)

# ── 3. Optimal constrained split per horizon ──
print(f"\n{'═'*90}")
print(f"  BEST SPLIT WITH ≤ ${MAX_MONTHLY:,}/mo CONSTRAINT (min ${min_to_mortgage/1000:.0f}K to mortgage)")
print(f"{'═'*90}")

opt_rows = []
for yr in horizons:
    sub = df_constrained[df_constrained["Horizon (yr)"] == yr].sort_values("Net Worth ($)", ascending=False)
    best = sub.iloc[0]
    runner = sub.iloc[1]
    opt_rows.append({
        "Horizon": f"Year {yr}",
        "Best: To Mortgage": f"${best['Total to Mortgage ($K)']:.0f}K",
        "Best: To Market": f"${best['To Market ($K)']:.0f}K",
        "Monthly Payment": f"${best['Monthly Payment ($)']:,.0f}",
        "DCA Savings": f"${best['DCA Savings ($/mo)']:,.0f}/mo",
        "Net Worth": f"${best['Net Worth ($)']:,.0f}",
        "vs S1": f"${best['vs S1 ($)']:+,.0f}",
        "Winner": "Constrained Split" if best["vs S1 ($)"] > 0 else "S1 (All Market)",
        "Runner-Up": f"${runner['Total to Mortgage ($K)']:.0f}K / ${runner['To Market ($K)']:.0f}K",
        "Runner-Up NW": f"${runner['Net Worth ($)']:,.0f}",
    })

df_opt_constrained = pd.DataFrame(opt_rows)
display(df_opt_constrained)

# ── 4. The "peace of mind" scenario: exactly ≤$5K/mo ──
peace_split_mtg = min_to_mortgage
peace_split_mkt = LUMP_SUM - peace_split_mtg
peace_dca = full_pmt - actual_pmt

print(f"\n{'━'*90}")
print(f"  PEACE-OF-MIND SCENARIO: ${peace_split_mtg/1000:.0f}K mortgage / ${peace_split_mkt/1000:.0f}K market")
print(f"  Monthly payment: ${actual_pmt:,.2f}  |  DCA savings: ${peace_dca:,.2f}/mo")
print(f"{'━'*90}")

peace_rows = []
for yr in [5, 10, 15, 20, 25, 30]:
    m = yr * 12
    mkt_val = peace_split_mkt * (1 + mkt_r) ** m
    mkt_val += peace_dca * ((1 + mkt_r) ** m - 1) / mkt_r

    remaining = remaining_mortgage * (1 + mort_r) ** m - actual_pmt * ((1 + mort_r) ** m - 1) / mort_r
    remaining = max(remaining, 0)
    nw = mkt_val + MORTGAGE - remaining
    s1_nw = s1_by_horizon[yr]

    peace_rows.append({
        "Year": yr,
        "Market Portfolio": f"${mkt_val:,.0f}",
        "Remaining Balance": f"${remaining:,.0f}",
        "Net Worth": f"${nw:,.0f}",
        "S1 Net Worth": f"${s1_nw:,.0f}",
        "Difference (vs S1)": f"${nw - s1_nw:+,.0f}",
        "Monthly Outgo": f"${actual_pmt:,.0f}",
    })

df_peace = pd.DataFrame(peace_rows)
display(df_peace)

# ── 5. Cost of peace of mind ──
yr30_peace = df_constrained[
    (df_constrained["Horizon (yr)"] == 30) &
    (df_constrained["Total to Mortgage ($K)"] == min_to_mortgage / 1000)
]["Net Worth ($)"].iloc[0]
yr30_s1 = s1_by_horizon[30]
cost_of_peace = yr30_s1 - yr30_peace

print(f"\n{'━'*90}")
print(f"  COST OF PEACE OF MIND (30-year horizon)")
print(f"{'━'*90}")
print(f"  S1 (All Market) Net Worth at Year 30:  ${yr30_s1:>14,.0f}")
print(f"  Peace-of-Mind Net Worth at Year 30:     ${yr30_peace:>14,.0f}")
print(f"  Cost of peace of mind:                  ${cost_of_peace:>14,.0f}")
print(f"  Monthly payment saved: ${full_pmt - actual_pmt:,.2f}/mo (invested as DCA)")
print(f"  That's ${cost_of_peace/30:,.0f}/year or ${cost_of_peace/360:,.0f}/month averaged over 30 years")
print(f"{'━'*90}")

# ── 6. Chart: Constrained splits vs S1 at each horizon ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, yr in enumerate(horizons):
    ax = axes[idx]
    sub = df_constrained[df_constrained["Horizon (yr)"] == yr].copy()
    ax.plot(sub["Total to Mortgage ($K)"], sub["Net Worth ($)"] / 1e6,
            "b-", lw=2, label="Constrained Split+DCA")
    ax.axhline(s1_by_horizon[yr] / 1e6, color="red", ls="--", lw=1.5, label="S1 (All Market)")
    ax.axvline(min_to_mortgage / 1000, color="green", ls=":", lw=1.5, alpha=0.7,
               label=f"Min for ≤${MAX_MONTHLY/1000:.0f}K/mo")

    best_row = sub.loc[sub["Net Worth ($)"].idxmax()]
    ax.plot(best_row["Total to Mortgage ($K)"], best_row["Net Worth ($)"] / 1e6,
            "g*", ms=15, zorder=5, label=f"Best: ${best_row['Total to Mortgage ($K)']:.0f}K")

    ax.set_title(f"Year {yr}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Total to Mortgage ($K)")
    ax.set_ylabel("Net Worth ($M)")
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.2f}M"))
    ax.legend(fontsize=7, loc="best")
    ax.grid(alpha=0.3)

fig.suptitle(f"Constrained Split+DCA (≤ ${MAX_MONTHLY:,}/mo) vs S1\n"
             f"Minimum ${min_to_mortgage/1000:.0f}K to mortgage | "
             f"Mortgage {MORT_RATE}% | Market {MKT_RATE}%",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
"""
Section 11 — Split + DCA Sensitivity Analysis
===============================================
For every possible split of $700K between market and mortgage (with DCA),
compare net worth against S1 (All Market) at multiple time horizons.
"""
import numpy as np
from matplotlib.colors import TwoSlopeNorm

# ── Helper: monthly payment on a loan ──
def _monthly_pmt(principal, r=mort_r, n=n_months):
    """Monthly payment for a fixed-rate loan."""
    if principal <= 0:
        return 0.0
    factor = (1 + r) ** n
    return principal * (r * factor) / (factor - 1)

# Full mortgage payment (baseline)
_full_pmt = _monthly_pmt(MORTGAGE)

# ── 1. Define the grid ──
split_step = 10_000                          # $10K increments
splits = list(range(0, LUMP_SUM + 1, split_step))
horizons = [5, 10, 15, 20, 25, 30]

# S1 net worth at each horizon: all $700K in market, full mortgage, no DCA
s1_by_horizon = {}
for yr in horizons:
    m = yr * 12
    mkt_val = LUMP_SUM * (1 + mkt_r) ** m
    bal = MORTGAGE * (1 + mort_r) ** m - _full_pmt * ((1 + mort_r) ** m - 1) / mort_r
    bal = max(bal, 0)
    s1_by_horizon[yr] = mkt_val + MORTGAGE - bal

# ── 2. Compute every split × horizon combination ──
grid_rows = []
for to_mtg in splits:
    to_mkt = LUMP_SUM - to_mtg
    reduced_loan = MORTGAGE - to_mtg
    reduced_pmt = _monthly_pmt(reduced_loan)
    dca = _full_pmt - reduced_pmt  # monthly savings reinvested

    for yr in horizons:
        m = yr * 12
        # Market portfolio: lump sum compounded + DCA annuity
        mkt_val = to_mkt * (1 + mkt_r) ** m
        if dca > 0:
            mkt_val += dca * ((1 + mkt_r) ** m - 1) / mkt_r

        # Mortgage balance at month m
        if reduced_loan > 0:
            bal = reduced_loan * (1 + mort_r) ** m - reduced_pmt * ((1 + mort_r) ** m - 1) / mort_r
            bal = max(bal, 0)
        else:
            bal = 0

        nw = mkt_val + MORTGAGE - bal
        grid_rows.append({
            "To Mortgage ($K)": to_mtg / 1000,
            "To Market ($K)": to_mkt / 1000,
            "DCA ($/mo)": round(dca, 2),
            "Horizon (yr)": yr,
            "Net Worth": round(nw, 2),
            "vs S1 ($)": round(nw - s1_by_horizon[yr], 2),
        })

df_grid = pd.DataFrame(grid_rows)

# ── 3. Pivot tables ──
pivot_nw = df_grid.pivot_table(index="To Mortgage ($K)", columns="Horizon (yr)",
                                values="Net Worth").round(0)
pivot_vs = df_grid.pivot_table(index="To Mortgage ($K)", columns="Horizon (yr)",
                                values="vs S1 ($)").round(0)

# Build df_sensitivity from pivot tables for display later
df_sensitivity = pd.DataFrame()
for yr in horizons:
    df_sensitivity[f"NW Yr{yr}"] = pivot_nw[yr]
    df_sensitivity[f"vs S1 Yr{yr}"] = pivot_vs[yr]
df_sensitivity.index = pivot_nw.index

# ── 4. Find optimal split for each horizon ──
optimal_rows = []
for yr in horizons:
    sub = df_grid[df_grid["Horizon (yr)"] == yr].sort_values("Net Worth", ascending=False)
    best = sub.iloc[0]
    runner = sub.iloc[1]
    optimal_rows.append({
        "Horizon": f"Year {yr}",
        "Best Split: To Market": f"${best['To Market ($K)']:.0f}K",
        "Best Split: To Mortgage": f"${best['To Mortgage ($K)']:.0f}K",
        "DCA ($/mo)": f"${best['DCA ($/mo)']:,.0f}",
        "Net Worth": f"${best['Net Worth']:,.0f}",
        "vs S1": f"${best['vs S1 ($)']:+,.0f}",
        "Winner": "SPLIT+DCA" if best["vs S1 ($)"] > 0 else "S1 (All Market)",
        "Runner-Up Split": f"${runner['To Market ($K)']:.0f}K / ${runner['To Mortgage ($K)']:.0f}K",
        "Runner-Up NW": f"${runner['Net Worth']:,.0f}",
        "Runner-Up vs S1": f"${runner['vs S1 ($)']:+,.0f}",
    })
df_optimal = pd.DataFrame(optimal_rows)

print("═" * 100)
print("  OPTIMAL SPLIT + DCA BY HORIZON")
print("═" * 100)
display(df_optimal)

# ── 5. Multi-panel chart: NW curves at each horizon ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, yr in enumerate(horizons):
    ax = axes[idx]
    sub = df_grid[df_grid["Horizon (yr)"] == yr].copy()
    ax.plot(sub["To Mortgage ($K)"], sub["Net Worth"] / 1e6, "b-", lw=2, label="Split+DCA")
    ax.axhline(s1_by_horizon[yr] / 1e6, color="red", ls="--", lw=1.5, label="S1 (All Market)")

    # Mark optimal
    best_row = sub.loc[sub["Net Worth"].idxmax()]
    ax.plot(best_row["To Mortgage ($K)"], best_row["Net Worth"] / 1e6,
            "g*", ms=15, zorder=5, label=f"Best: ${best_row['To Mortgage ($K)']:.0f}K")

    ax.set_title(f"Year {yr}", fontsize=12, fontweight="bold")
    ax.set_xlabel("To Mortgage ($K)")
    ax.set_ylabel("Net Worth ($M)")
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.2f}M"))
    ax.legend(fontsize=8, loc="best")
    ax.grid(alpha=0.3)

fig.suptitle(f"Split + DCA Net Worth vs S1 (All Market) at Each Horizon\n"
             f"Mortgage {MORT_RATE}% | Market {MKT_RATE}% | {MORT_YEARS}yr",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── 6. Display the full sensitivity table (every 50K) ──
print(f"\n{'━'*100}")
print("  FULL SENSITIVITY TABLE (every $50K split)")
print(f"{'━'*100}\n")

df_show = df_sensitivity.loc[
    df_sensitivity.index.isin(
        [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700]
    )
].copy()

fmt_show = df_show.copy()
for col in fmt_show.columns:
    if "vs S1" in col:
        fmt_show[col] = fmt_show[col].apply(lambda x: f"${x:+,.0f}" if x != 0 else "← S1")
    else:
        fmt_show[col] = fmt_show[col].apply(lambda x: f"${x:,.0f}")
display(fmt_show)

# ── 7. Heatmap: vs S1 advantage by split × horizon ──
fig, ax = plt.subplots(figsize=(14, 8))

heatmap_data = df_grid.pivot_table(
    index="To Mortgage ($K)",
    columns="Horizon (yr)",
    values="vs S1 ($)",
).round(0)

hm_50k = heatmap_data.loc[heatmap_data.index.isin(list(range(0, 701, 50)))]

v_abs = max(abs(hm_50k.values.min()), abs(hm_50k.values.max()), 1)
norm = TwoSlopeNorm(vcenter=0, vmin=-v_abs, vmax=v_abs)

im = ax.imshow(hm_50k.values, aspect="auto", cmap="RdYlGn", norm=norm, origin="lower")
ax.set_xticks(range(len(hm_50k.columns)))
ax.set_xticklabels([f"Yr {c}" for c in hm_50k.columns], fontsize=11)
ax.set_yticks(range(len(hm_50k.index)))
ax.set_yticklabels([f"${int(v)}K" for v in hm_50k.index], fontsize=10)

for i in range(len(hm_50k.index)):
    for j in range(len(hm_50k.columns)):
        val = hm_50k.values[i, j]
        color = "white" if abs(val) > v_abs * 0.6 else "black"
        ax.text(j, i, f"${val/1000:+,.0f}K", ha="center", va="center",
                fontsize=9, fontweight="bold", color=color)

fig.colorbar(im, ax=ax, label="$ Advantage vs S1", shrink=0.8)
ax.set_xlabel("Time Horizon", fontsize=12)
ax.set_ylabel("Amount to Mortgage ($K)", fontsize=12)
ax.set_title(f"Advantage of Split+DCA vs S1 (All Market)\n"
             f"Green = Split wins  |  Red = S1 wins  |  Mortgage {MORT_RATE}% vs Market {MKT_RATE}%",
             fontsize=13)
plt.tight_layout()
plt.show()

# ── 8. Key insight ──
print(f"\n{'━'*100}")
print("  KEY INSIGHTS")
print(f"{'━'*100}")
for yr in horizons:
    sub = df_grid[df_grid["Horizon (yr)"] == yr]
    best_row = sub.loc[sub["Net Worth"].idxmax()]
    s1_nw = s1_by_horizon[yr]
    advantage = best_row["Net Worth"] - s1_nw
    if advantage > 0:
        print(f"  Year {yr:2d}: SPLIT WINS — Best ${best_row['To Mortgage ($K)']:.0f}K/"
              f"${best_row['To Market ($K)']:.0f}K → +${advantage:,.0f} vs S1")
    else:
        print(f"  Year {yr:2d}: S1 WINS   — All-market advantage: ${-advantage:,.0f}")

mkt_advantage_yr30 = s1_by_horizon[30] - df_grid[
    (df_grid["Horizon (yr)"] == 30) & (df_grid["To Mortgage ($K)"] == 700)
]["Net Worth"].iloc[0]
print(f"\n  At Year 30: S1 (All Market) vs S3 (All Mortgage+DCA) gap = ${mkt_advantage_yr30:,.0f}")
print(f"  Rate spread: Market {MKT_RATE}% − Mortgage {MORT_RATE}% = {MKT_RATE - MORT_RATE:.1f}%")
if MKT_RATE > MORT_RATE:
    print(f"  Since market ({MKT_RATE}%) > mortgage ({MORT_RATE}%), more market allocation = higher long-term NW")
else:
    print(f"  Since mortgage ({MORT_RATE}%) ≥ market ({MKT_RATE}%), mortgage paydown + DCA can win!")
print(f"{'━'*100}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 13 — MASTER EXCEL EXPORT: ALL RESULTS IN ONE FILE
# ══════════════════════════════════════════════════════════════════════════════
import os
from openpyxl.styles import Font, numbers

output_path = os.path.join(r"I:\masterswork\FinanceData", "all_mortgage_market_analysis.xlsx")
header_font = Font(bold=True)
dollar_fmt  = '#,##0'

# === Helper (ensure available) ===
def _monthly_pmt_ex(principal, r=mort_r, n=n_months):
    if principal <= 0:
        return 0.0
    factor = (1 + r) ** n
    return principal * (r * factor) / (factor - 1)

# Re-derive the optimal allocation row (best was overwritten by later sections)
best_alloc = df_alloc.loc[df_alloc["net_worth_30yr"].idxmax()]

# ═══════════════════════════════════════════════════════════════
#  PREPARE ALL DATAFRAMES
# ═══════════════════════════════════════════════════════════════

# ── 1. Scenario Summary ──
scenario_summary = pd.DataFrame([
    {"Strategy": "S1 — All Market",
     "To Mortgage ($)": s1["to_mortgage"], "To Market ($)": s1["to_market"],
     "Reduced Loan ($)": s1["reduced_loan"], "Monthly Payment ($)": s1["monthly_payment"],
     "Monthly Saving ($)": s1["monthly_saving"], "Monthly DCA ($)": s1["monthly_dca"],
     "Reinvest?": "Yes" if s1["reinvest_savings"] else "No",
     "Total Interest 30yr ($)": s1["total_interest"],
     "Market Portfolio 30yr ($)": s1["market_value_30yr"],
     "Home Equity ($)": s1["equity"], "Net Worth 30yr ($)": s1["net_worth_30yr"]},
    {"Strategy": "S2 — All Mortgage (no DCA)",
     "To Mortgage ($)": s2["to_mortgage"], "To Market ($)": s2["to_market"],
     "Reduced Loan ($)": s2["reduced_loan"], "Monthly Payment ($)": s2["monthly_payment"],
     "Monthly Saving ($)": s2["monthly_saving"], "Monthly DCA ($)": s2["monthly_dca"],
     "Reinvest?": "Yes" if s2["reinvest_savings"] else "No",
     "Total Interest 30yr ($)": s2["total_interest"],
     "Market Portfolio 30yr ($)": s2["market_value_30yr"],
     "Home Equity ($)": s2["equity"], "Net Worth 30yr ($)": s2["net_worth_30yr"]},
    {"Strategy": "S3 — Mortgage + DCA",
     "To Mortgage ($)": s3["to_mortgage"], "To Market ($)": s3["to_market"],
     "Reduced Loan ($)": s3["reduced_loan"], "Monthly Payment ($)": s3["monthly_payment"],
     "Monthly Saving ($)": s3["monthly_saving"], "Monthly DCA ($)": s3["monthly_dca"],
     "Reinvest?": "Yes" if s3["reinvest_savings"] else "No",
     "Total Interest 30yr ($)": s3["total_interest"],
     "Market Portfolio 30yr ($)": s3["market_value_30yr"],
     "Home Equity ($)": s3["equity"], "Net Worth 30yr ($)": s3["net_worth_30yr"]},
    {"Strategy": f"S5 — ${S5_TO_MARKET:,.0f} Mkt + ${S5_TO_MORTGAGE:,.0f} Mtg",
     "To Mortgage ($)": s5["to_mortgage"], "To Market ($)": s5["to_market"],
     "Reduced Loan ($)": s5["reduced_loan"], "Monthly Payment ($)": s5["monthly_payment"],
     "Monthly Saving ($)": s5["monthly_saving"], "Monthly DCA ($)": s5["monthly_dca"],
     "Reinvest?": "Yes" if s5["reinvest_savings"] else "No",
     "Total Interest 30yr ($)": s5["total_interest"],
     "Market Portfolio 30yr ($)": s5["market_value_30yr"],
     "Home Equity ($)": s5["equity"], "Net Worth 30yr ($)": s5["net_worth_30yr"]},
    {"Strategy": f"Opt — ${best_alloc['to_mortgage']:,.0f} split",
     "To Mortgage ($)": best_alloc["to_mortgage"], "To Market ($)": best_alloc["to_market"],
     "Reduced Loan ($)": best_alloc["reduced_loan"], "Monthly Payment ($)": best_alloc["monthly_payment"],
     "Monthly Saving ($)": best_alloc["monthly_saving"], "Monthly DCA ($)": best_alloc["monthly_dca"],
     "Reinvest?": "Yes" if best_alloc["reinvest_savings"] else "No",
     "Total Interest 30yr ($)": best_alloc["total_interest"],
     "Market Portfolio 30yr ($)": best_alloc["market_value_30yr"],
     "Home Equity ($)": best_alloc["equity"], "Net Worth 30yr ($)": best_alloc["net_worth_30yr"]},
]).set_index("Strategy")

# ── 2. Allocation Sweep ──
df_alloc_export = df_alloc.rename(columns={
    "to_mortgage": "To Mortgage ($)", "to_market": "To Market ($)",
    "reduced_loan": "Reduced Loan ($)", "monthly_payment": "Monthly Payment ($)",
    "monthly_saving": "Monthly Saving ($)", "monthly_dca": "Monthly DCA ($)",
    "reinvest_savings": "Reinvest?", "total_interest": "Total Interest ($)",
    "market_value_30yr": "Market Value 30yr ($)", "remaining_balance": "Remaining Balance ($)",
    "equity": "Home Equity ($)", "net_worth_30yr": "Net Worth 30yr ($)",
})

# ── 3. Year-by-Year Milestones (raw NW columns — formulas added later) ──
df_mile_export = df_mile.drop(columns=["S1_vs_S2", "S1_vs_S3", "S1_vs_S5", "S3_vs_S2"])
df_mile_export = df_mile_export.rename(columns={
    "S1_NW": "S1 (All Market) NW", "S1_Market": "S1 Market Portfolio", "S1_Interest": "S1 Interest Paid",
    "S2_NW": "S2 (All Mortgage) NW", "S2_Market": "S2 Market Portfolio", "S2_Interest": "S2 Interest Paid",
    "S3_NW": "S3 (Mortgage+DCA) NW", "S3_Market": "S3 Market Portfolio", "S3_Interest": "S3 Interest Paid",
    "S5_NW": "S5 ($500K+$200K) NW", "S5_Market": "S5 Market Portfolio", "S5_Interest": "S5 Interest Paid",
    "Opt_NW": "Optimal Split NW", "Opt_Market": "Optimal Market Portfolio",
})

# ── 4. Opportunity Loss (raw NW + Winner — formulas for losses added later) ──
df_opp_export = df_winners[["S1_NW", "S2_NW", "S3_NW", "S5_NW", "Opt_NW", "Winner"]].copy()
df_opp_export = df_opp_export.rename(columns={
    "S1_NW": "S1 (All Market) NW", "S2_NW": "S2 (All Mortgage) NW",
    "S3_NW": "S3 (Mortgage+DCA) NW", "S5_NW": "S5 ($500K+$200K) NW",
    "Opt_NW": "Opt (Best Split) NW",
})

# ── 5. Base Mortgage Annual ──
annual_export = annual.rename(columns={
    "Payments": "# Payments", "Total_Payment": "Total Payment ($)",
    "Total_Principal": "Total Principal ($)", "Total_Interest": "Total Interest ($)",
    "End_Balance": "End of Year Balance ($)",
})

# ── 6 & 7. Comparative Annual (formulas for Opp Cost added later) ──
comp_col_map = {
    "Interest_Paid": "Interest Paid ($)", "Principal_Paid": "Principal Paid ($)",
    "Cumul_Interest": "Cumulative Interest ($)", "Equity": "Equity Built ($)",
    "Interest_Invested": "Interest If Invested @7% ($)",
    "FullPmt_Invested": "Full Pmt If Invested @7% ($)", "Balance": "Remaining Balance ($)",
}
annual_a_export = annual_a.drop(columns=["Opp_Cost_Interest", "Opp_Cost_Full"]).rename(columns=comp_col_map)
annual_b_export = annual_b.drop(columns=["Opp_Cost_Interest", "Opp_Cost_Full"]).rename(columns=comp_col_map)

# ── 8. Sensitivity Grid (NW + vs S1 at every $50K split × 6 horizons) ──
# df_sensitivity built in Section 11
df_sens_export = df_sensitivity.copy()
df_sens_export.index.name = "To Mortgage ($K)"

# ── 9. Optimal Split by Horizon ──
# df_optimal from Section 11

# ── 10. Constrained Split (≤$5K/mo) by Horizon ──
# df_opt_constrained from Section 12

# ── 11. Peace-of-Mind Year-by-Year ──
# df_peace from Section 12

# ═══════════════════════════════════════════════════════════════
#  WRITE TO EXCEL + ADD FORMULAS
# ═══════════════════════════════════════════════════════════════

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # --- Write all sheets ---
    scenario_summary.to_excel(writer,     sheet_name="1 Scenario Summary")
    df_alloc_export.to_excel(writer,      sheet_name="2 Allocation Sweep", index=False)
    df_mile_export.to_excel(writer,       sheet_name="3 Year-by-Year NW")
    df_opp_export.to_excel(writer,        sheet_name="4 Opportunity Loss")
    df_ms.to_excel(writer,                sheet_name="5 Key Milestones", index=False)
    df_winner_summary.to_excel(writer,    sheet_name="6 Milestone Winners")
    annual_export.to_excel(writer,        sheet_name="7 Base Mortgage Annual")
    annual_a_export.to_excel(writer,      sheet_name="8 Comparative 1M Annual")
    annual_b_export.to_excel(writer,      sheet_name="9 Comparative 1.7M Annual")
    df_sens_export.to_excel(writer,       sheet_name="10 Sensitivity Grid")
    df_optimal.to_excel(writer,           sheet_name="11 Optimal Split by Horizon", index=False)
    df_opt_constrained.to_excel(writer,   sheet_name="12 Constrained 5K Split", index=False)
    df_peace.to_excel(writer,             sheet_name="13 Peace of Mind Detail", index=False)

    # --- Also export the full constrained sweep (raw data) ---
    df_constrained_pivot = df_constrained.pivot_table(
        index="Total to Mortgage ($K)",
        columns="Horizon (yr)",
        values="Net Worth ($)",
    ).round(0)
    df_constrained_pivot.to_excel(writer, sheet_name="14 Constrained NW Grid")

    # ─── FORMULAS: Sheet 3 — Year-by-Year NW ───
    # A=Year, B=S1 NW, C=S1 Mkt, D=S1 Int, E=S2 NW, F=S2 Mkt, G=S2 Int,
    #         H=S3 NW, I=S3 Mkt, J=S3 Int, K=S5 NW, L=S5 Mkt, M=S5 Int,
    #         N=Opt NW, O=Opt Mkt
    ws = writer.sheets["3 Year-by-Year NW"]
    for col_letter, hdr in [("P", "S1 vs S2 (+= S1 Wins)"), ("Q", "S1 vs S3 (+= S1 Wins)"),
                             ("R", "S1 vs S5 (+= S1 Wins)"), ("S", "S3 vs S2 (DCA Value)"),
                             ("T", "Best NW (MAX)"), ("U", "Winner")]:
        ws[f"{col_letter}1"].value = hdr
        ws[f"{col_letter}1"].font = header_font
    for row in range(2, 32):
        ws[f"P{row}"] = f"=B{row}-E{row}"
        ws[f"Q{row}"] = f"=B{row}-H{row}"
        ws[f"R{row}"] = f"=B{row}-K{row}"
        ws[f"S{row}"] = f"=H{row}-E{row}"
        ws[f"T{row}"] = f"=MAX(B{row},E{row},H{row},K{row},N{row})"
        ws[f"U{row}"] = (
            f'=IF(B{row}=T{row},"S1 (All Market)",'
            f'IF(E{row}=T{row},"S2 (All Mortgage)",'
            f'IF(H{row}=T{row},"S3 (Mortgage+DCA)",'
            f'IF(K{row}=T{row},"S5 ($500K+$200K)",'
            f'"Opt (Best Split)"))))'
        )
        for c in "PQRST":
            ws[f"{c}{row}"].number_format = dollar_fmt

    # ─── FORMULAS: Sheet 4 — Opportunity Loss ───
    # A=Year, B=S1 NW, C=S2 NW, D=S3 NW, E=S5 NW, F=Opt NW, G=Winner
    ws2 = writer.sheets["4 Opportunity Loss"]
    for col_letter, hdr in [("H", "Best NW (MAX)"), ("I", "S1 Loss"), ("J", "S2 Loss"),
                             ("K", "S3 Loss"), ("L", "S5 Loss"), ("M", "Opt Loss")]:
        ws2[f"{col_letter}1"].value = hdr
        ws2[f"{col_letter}1"].font = header_font
    for row in range(2, 32):
        ws2[f"H{row}"] = f"=MAX(B{row}:F{row})"
        for i, c in enumerate("IJKLM"):
            src = chr(ord("B") + i)
            ws2[f"{c}{row}"] = f"={src}{row}-H{row}"
            ws2[f"{c}{row}"].number_format = dollar_fmt
        ws2[f"H{row}"].number_format = dollar_fmt

    # ─── FORMULAS: Sheets 8 & 9 — Comparative Annual Opp Cost ───
    for sn in ["8 Comparative 1M Annual", "9 Comparative 1.7M Annual"]:
        ws3 = writer.sheets[sn]
        ws3["I1"].value = "Opp Cost — Interest ($)"
        ws3["I1"].font = header_font
        ws3["J1"].value = "Opp Cost — Full Payment ($)"
        ws3["J1"].font = header_font
        for row in range(2, 32):
            ws3[f"I{row}"] = f"=F{row}-D{row}"
            ws3[f"J{row}"] = f"=G{row}-E{row}"
            ws3[f"I{row}"].number_format = dollar_fmt
            ws3[f"J{row}"].number_format = dollar_fmt

    # ─── AUTO-FIT COLUMN WIDTHS (approximate) ───
    for sn in writer.sheets:
        ws_fit = writer.sheets[sn]
        for col_cells in ws_fit.columns:
            max_len = 0
            col_letter = col_cells[0].column_letter
            for cell in col_cells:
                try:
                    val = str(cell.value) if cell.value else ""
                    max_len = max(max_len, len(val))
                except:
                    pass
            ws_fit.column_dimensions[col_letter].width = min(max_len + 3, 30)

print(f"✅ Exported 14 sheets to:\n   {output_path}")
print()
sheets = [
    ("1  Scenario Summary",         "S1/S2/S3/S5/Opt side-by-side (30yr)"),
    ("2  Allocation Sweep",         "Full $0–$700K sweep (71 rows)"),
    ("3  Year-by-Year NW",          "30-yr milestones + formulas: diffs, MAX, Winner"),
    ("4  Opportunity Loss",         "NW per strategy + formula: Loss = NW − MAX"),
    ("5  Key Milestones",           "5/10/15 yr detailed breakdown"),
    ("6  Milestone Winners",        "Winner summary at key horizons"),
    ("7  Base Mortgage Annual",     f"$1.7M @ {ANNUAL_RATE_PCT}% annual schedule"),
    ("8  Comparative 1M Annual",    "Opp cost formulas: Invested − Paid"),
    ("9  Comparative 1.7M Annual",  "Opp cost formulas: Invested − Equity"),
    ("10 Sensitivity Grid",         "NW + vs S1 at every $50K split × 6 horizons"),
    ("11 Optimal Split by Horizon", "Best split+DCA at 5/10/15/20/25/30 yr + runner-up"),
    ("12 Constrained 5K Split",     f"Best split with <=${MAX_MONTHLY:,}/mo payment"),
    ("13 Peace of Mind Detail",     f"Year-by-year for ${min_to_mortgage/1000:.0f}K/$"
                                    f"{(LUMP_SUM-min_to_mortgage)/1000:.0f}K split"),
    ("14 Constrained NW Grid",      "Full constrained sweep pivot table"),
]
for name, desc in sheets:
    print(f"   {name:<35s} — {desc}")